# 03 - Optimization Model

**Purpose:** Run LP optimization for battery dispatch and visualize optimal schedule.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config.settings import load_config
from src.data.loaders import generate_synthetic_data
from src.optimization.scheduler import MicrogridScheduler
from src.evaluation.baseline import compute_baseline_cost
from src.evaluation.metrics import compute_savings_metrics, format_metrics_report
from src.plots.visualizations import (
    set_plot_style,
    plot_dispatch_schedule,
    plot_soc_trajectory,
    plot_cost_comparison,
)

%matplotlib inline
set_plot_style()

## Load Configuration and Data

In [ ]:
settings = load_config('../config/config.yaml')

print("Battery Configuration:")
print(f"  Capacity: {settings.battery.capacity_kwh} kWh")
print(f"  Max Power: {settings.battery.max_power_kw} kW")
print(f"  Charge Efficiency: {settings.battery.efficiency_charge}")
print(f"  Discharge Efficiency: {settings.battery.efficiency_discharge}")
print(f"  SOC Range: {settings.battery.soc_min*100:.0f}% - {settings.battery.soc_max*100:.0f}%")

In [ ]:
df = generate_synthetic_data(
    start_date=settings.start_date,
    end_date=settings.end_date,
    resolution_minutes=settings.resolution_minutes,
    solar_capacity_kw=settings.solar_capacity_kw,
    timezone=settings.timezone,
)
print(f"Time horizon: {len(df)} steps ({len(df)} hours)")

## Run Optimization

In [ ]:
scheduler = MicrogridScheduler(settings)

result = scheduler.optimize(
    timestamps=df.index,
    prices=df['price'].values,
    p_solar=df['solar'].values,
    p_load=df['load'].values,
)

print(f"Solver Status: {result.solve_status}")
print(f"Solve Time: {result.solve_time:.3f} seconds")
print(f"Optimal Cost: ${result.total_cost:.2f}")

## Compare with Baseline

In [ ]:
dt_hours = settings.resolution_minutes / 60.0

baseline = compute_baseline_cost(
    prices=df['price'].values,
    p_solar=df['solar'].values,
    p_load=df['load'].values,
    feed_in_tariff=settings.grid.feed_in_tariff,
    dt_hours=dt_hours,
)

metrics = compute_savings_metrics(
    baseline=baseline,
    optimized=result,
    battery_capacity_kwh=settings.battery.capacity_kwh,
    dt_hours=dt_hours,
)

print(format_metrics_report(metrics))

## Visualize Dispatch Schedule

In [ ]:
fig = plot_dispatch_schedule(result)
plt.show()

## State of Charge Trajectory

In [ ]:
fig = plot_soc_trajectory(
    result,
    soc_min=settings.battery.soc_min,
    soc_max=settings.battery.soc_max,
)
plt.show()

## Cost Comparison

In [ ]:
fig = plot_cost_comparison(baseline.total_cost, result.total_cost)
plt.show()

## Optimization Strategy Analysis

The optimization exhibits price arbitrage behavior:
- **Charging** during low-price periods (overnight, midday solar)
- **Discharging** during high-price periods (afternoon/evening peak)

In [ ]:
# Analyze charging/discharging vs price
fig, ax = plt.subplots(figsize=(12, 5))

ax.scatter(result.prices, result.p_charge, alpha=0.6, label='Charging', color='green')
ax.scatter(result.prices, result.p_discharge, alpha=0.6, label='Discharging', color='red')

ax.set_xlabel('Price ($/kWh)')
ax.set_ylabel('Power (kW)')
ax.set_title('Battery Action vs Electricity Price')
ax.legend()

plt.tight_layout()
plt.show()